# Data Preparation

## Current Problem

Our data has an imbalance of 94:6 (no claim : claim)

We want to have a similar amount of labels or classes that would help us identify claim and no claim. Having the majority as claim would not give enough data to our model to how a "no claim" policy holder would look like. 

Hence, we employ data-level techniques such as:

- SMOTE (Synthetic Minority Oversampling Technique)
    - Creates synthetic samples by interpolating between existing minority samples and their neighbors

- Random Undersampling

    - Removes majority class samples randomly

- Tomek Links
    - Remove pairs of examples from opposite classes that are nearest neighbors
    



In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, precision_recall_curve, average_precision_score,
    roc_curve, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline
import kagglehub
import os


In [7]:
path = kagglehub.dataset_download("litvinenko630/insurance-claims")

# List all files in the downloaded directory
files = os.listdir(path)
file_path = os.path.join(path, "Insurance claims data.csv")
df = pd.read_csv(file_path)

In [8]:
from helpers.preprocess import *

df_processed = preprocess_data(df)
print(f"Processed shape: {df_processed.shape}")
print(f"\nFeature types after processing:")
print(df_processed.dtypes.value_counts())

Processed shape: (58592, 58677)

Feature types after processing:
bool       58660
int64         10
float64        7
Name: count, dtype: int64


In [9]:
# Separate features and target
X = df_processed.drop('claim_status', axis=1)
y = df_processed['claim_status']

# Train-test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(y_train.value_counts())

Training set: 46873 samples
Test set: 11719 samples

Training class distribution:
claim_status
0    43875
1     2998
Name: count, dtype: int64


## SMOTE

1. For each minority sample, find its k nearest neighbors (default k=5)
2. Randomly select one of those neighbors
3. Create a new synthetic sample along the line connecting them

Formula: `new_sample = original + random(0,1) × (neighbor - original)` 

This creates realistic variations that help the model generalize better.

### Also has some Variants
#### BorderlineSMOTE
Only generates synthetic samples for minority instances that are **near the decision boundary** (close to majority samples). This focuses on the hard-to-classify cases.

#### ADASYN (Adaptive Synthetic Sampling)
Generates **more synthetic samples** for minority instances that are harder to classify (surrounded by majority samples). It's adaptive to the local density.


In [ ]:
# Scale features to standardize them
# mean = 0, sigma = 1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

: 

In [ ]:
# Apply SMOTE to our insurance data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("SMOTE Results:")
print(f"  Original training set: {len(y_train)} samples")
print(f"  After SMOTE: {len(y_train_smote)} samples")
print(f"  \nClass distribution after SMOTE:")
print(f"  No Claim: {sum(y_train_smote == 0)}")
print(f"  Claim: {sum(y_train_smote == 1)}")

In [ ]:
# Apply BorderlineSMOTE to our insurance data
smote = BorderlineSMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("BorderlineSMOTE Results:")
print(f"  Original training set: {len(y_train)} samples")
print(f"  After BorderlineSMOTE: {len(y_train_smote)} samples")
print(f"  \nClass distribution after BorderlineSMOTE:")
print(f"  No Claim: {sum(y_train_smote == 0)}")
print(f"  Claim: {sum(y_train_smote == 1)}")

In [ ]:
# Apply ADASYN to our insurance data
smote = ADASYN(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("ADASYN Results:")
print(f"  Original training set: {len(y_train)} samples")
print(f"  After ADASYN: {len(y_train_smote)} samples")
print(f"  \nClass distribution after ADASYN:")
print(f"  No Claim: {sum(y_train_smote == 0)}")
print(f"  Claim: {sum(y_train_smote == 1)}")

In [ ]:
# SMOTE + Tomek Links
smote_tomek = SMOTETomek(random_state=42)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train_scaled, y_train)

print("SMOTE + Tomek Links Results:")
print(f"  Original: {len(y_train)} samples")
print(f"  After SMOTE: ~{sum(y_train==0) + sum(y_train==0)} (balanced)")
print(f"  After Tomek cleanup: {len(y_train_st)} samples")
print(f"  \nClass distribution:")
print(f"  No Claim: {sum(y_train_st == 0)}")
print(f"  Claim: {sum(y_train_st == 1)}")